In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, KBinsDiscretizer, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

# ==========================================
# 1. Load Original Data
# ==========================================
original_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

y_train = original_df['Revenue']
X_train_raw = original_df.drop(columns=['Revenue', 'Session_ID'], errors='ignore')
X_test_raw = test_df.drop(columns=['Session_ID'], errors='ignore')

# ==========================================
# 2. Impute Missing Values Safely
# ==========================================
numeric_features = X_train_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_raw.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()

train_medians = X_train_raw[numeric_features].median()
train_modes = X_train_raw[categorical_features].mode().iloc[0]

X_train_imp = X_train_raw.copy()
X_train_imp[numeric_features] = X_train_imp[numeric_features].fillna(train_medians)
X_train_imp[categorical_features] = X_train_imp[categorical_features].fillna(train_modes)

X_test_imp = X_test_raw.copy()
X_test_imp[numeric_features] = X_test_imp[numeric_features].fillna(train_medians)
X_test_imp[categorical_features] = X_test_imp[categorical_features].fillna(train_modes)

# ==========================================
# 3. Explicit Feature Transformations
# ==========================================
def create_step_features(df):
    df = df.copy()
    eps = 1e-6
    
    # 1. Critical Binary Indicators
    if 'PageValues' in df.columns:
        df['PV_is_zero'] = (df['PageValues'] == 0).astype(int)
        df['PV_gt_0'] = (df['PageValues'] > 0).astype(int)
        df['PV_gt_10'] = (df['PageValues'] > 10).astype(int)
        
    if 'BounceRates' in df.columns:
        df['Zero_Bounce'] = (df['BounceRates'] == 0).astype(int)
        
    if 'ExitRates' in df.columns:
        df['Low_Exit'] = (df['ExitRates'] < 0.02).astype(int)

    # 2. Aggregations & Non-linear Ratios
    page_cols = [c for c in ['Administrative', 'Informational', 'ProductRelated'] if c in df.columns]
    dur_cols = [c for c in ['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration'] if c in df.columns]

    if page_cols:
        df['Total_Pages'] = df[page_cols].sum(axis=1)
    if dur_cols:
        df['Total_Duration'] = df[dur_cols].sum(axis=1)

    if 'PageValues' in df.columns and 'ExitRates' in df.columns:
        df['PV_per_Exit'] = df['PageValues'] / (df['ExitRates'] + eps)

    if 'Total_Duration' in df.columns and 'Total_Pages' in df.columns:
        df['Time_per_Page'] = df['Total_Duration'] / (df['Total_Pages'] + eps)

    return df

X_train_fe = create_step_features(X_train_imp)
X_test_fe = create_step_features(X_test_imp)

# Define column subgroups for target preprocessing
binned_cols = [c for c in ['PageValues', 'ExitRates', 'BounceRates', 'Total_Duration'] if c in X_train_fe.columns]
std_num_cols = [c for c in X_train_fe.select_dtypes(include=['int64', 'float64']).columns if c not in binned_cols]
cat_cols = X_train_fe.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()

# ==========================================
# 4. Column Transformer Pipeline
# ==========================================
preprocessor = ColumnTransformer(
    transformers=[
        ('num_std', StandardScaler(), std_num_cols),
        ('num_bin', KBinsDiscretizer(n_bins=5, encode='onehot-dense', strategy='quantile'), binned_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])

X_train_processed_array = preprocessor.fit_transform(X_train_fe)
feature_names = preprocessor.get_feature_names_out()

# Reconstruct processed_df to fulfill submission checks
processed_df = pd.DataFrame(X_train_processed_array, columns=feature_names)
processed_df['Revenue'] = y_train.values

X_processed = processed_df.drop(columns=['Revenue'])
y_processed = processed_df['Revenue']

# ==========================================
# 5. Model Training & 10-Fold OOF Predictions
# ==========================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

oof_probs = np.zeros(len(X_processed))
X_test_processed_array = preprocessor.transform(X_test_fe)
X_test_processed = pd.DataFrame(X_test_processed_array, columns=feature_names)
test_probs_folds = np.zeros(len(X_test_processed))

# Fine-tuned L2 Logistic Regression with Optimized C Penalty
for fold, (train_idx, val_idx) in enumerate(skf.split(X_processed, y_processed)):
    X_tr, X_val = X_processed.iloc[train_idx], X_processed.iloc[val_idx]
    y_tr, y_val = y_processed.iloc[train_idx], y_processed.iloc[val_idx]
    
    fold_model = LogisticRegression(C=0.8, max_iter=3000, random_state=42 + fold)
    fold_model.fit(X_tr, y_tr)
    
    oof_probs[val_idx] = fold_model.predict_proba(X_val)[:, 1]
    test_probs_folds += fold_model.predict_proba(X_test_processed)[:, 1] / 10.0

# Precise Threshold Sweep for Pure Accuracy Maximization
thresholds = np.linspace(0.40, 0.60, 101)
best_threshold = 0.5
best_acc = 0.0

for t in thresholds:
    acc = accuracy_score(y_processed, oof_probs >= t)
    if acc > best_acc:
        best_acc = acc
        best_threshold = t

print(f"Optimal Threshold: {best_threshold:.4f}")
print(f"Estimated 10-Fold Cross-Validation Accuracy: {best_acc * 100:.3f}%")

# Primary model instance for checkpoint validation
model = LogisticRegression(C=0.8, max_iter=3000, random_state=42)
model.fit(X_processed, y_processed)

# Generate final predictions using ensembled probabilities and optimal threshold
predictions = (test_probs_folds >= best_threshold)

# ==========================================
# 6. Generate Official Submission File
# ==========================================
original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (processed_rows / original_rows) * 100

final_model = model
if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_
if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__

checkpoints = pd.DataFrame({
    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],
    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})

prediction_output = pd.DataFrame({
    "id": test_df["Session_ID"].astype(str),
    "value": np.asarray(predictions).astype(str)
})

submission = pd.concat(
    [checkpoints, prediction_output],
    ignore_index=True
)

submission.to_csv("submission4.csv", index=False)

print("\n--- SUCCESS ---")
print("submission.csv created successfully.")
print("\nCheckpoints Summary:")
print(checkpoints)

c:\Users\daksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
c:\Users\daksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\daksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 2 are removed. Consider decreasing the number of bins.
  warnings.warn(


Optimal Threshold: 0.5180
Estimated 10-Fold Cross-Validation Accuracy: 88.504%

--- SUCCESS ---
submission.csv created successfully.

Checkpoints Summary:
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  50
6  row_retained_percent               100.0
7            model_name  LogisticRegression
